# FabricEye QC: train a fabric defect detector (YOLO)

Runs on **Kaggle** (Settings: GPU T4 x2 or P100, Internet ON) or **Google Colab** (Runtime: T4 GPU).
At the end you download `best.pt` and put it in the app folder as `models/best.pt`.

**Dataset:** Fabric Defects Object Detection Dataset (Neamah, Figshare 2024): 720 colored fabric images,
4 classes: Oil, Hole, Cutting, Crack, already in YOLO format.
Two other options are included if the download fails.

In [ ]:
!pip install -q ultralytics py7zr

## 1. Get the dataset
Run **one** of the options below.

In [ ]:
# Option A: Figshare FD dataset (direct download, needs Internet ON)
import os, zipfile, glob, py7zr
WORK = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
DL = f"{WORK}/fd_raw"; os.makedirs(DL, exist_ok=True)
!wget -q -O {DL}/fd.zip "https://figshare.com/ndownloader/articles/25546465/versions/2"
with zipfile.ZipFile(f"{DL}/fd.zip") as z: z.extractall(DL)
for f in glob.glob(f"{DL}/**/*.7z", recursive=True):
    print("extracting", f)
    with py7zr.SevenZipFile(f, "r") as a: a.extractall(DL)
for f in glob.glob(f"{DL}/**/*.zip", recursive=True):
    if not f.endswith("fd.zip"):
        with zipfile.ZipFile(f) as z: z.extractall(os.path.dirname(f))
DATA_ROOT = DL
print("done")

In [ ]:
# Option B: any Roboflow Universe fabric dataset (free API key from roboflow.com)
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_KEY")
# ds = rf.workspace("sony-ba647").project("fabric-defect-detection-x6dcy").version(1).download("yolov8")
# DATA_ROOT = ds.location

In [ ]:
# Option C: a dataset you added on Kaggle with "Add Input" (or uploaded manually)
# DATA_ROOT = "/kaggle/input/your-dataset-folder"
# If you downloaded FD_Dataset.7z by hand from Figshare, upload it as a Kaggle dataset and point here.

## 2. Find the YOLO folders and write `data.yaml`
Works with most YOLO dataset layouts.

In [ ]:
import os, glob, yaml
from pathlib import Path

root = Path(DATA_ROOT)
yamls = [p for p in root.rglob("*.yaml") if "names" in open(p, errors="ignore").read()]
print("yaml files found:", yamls)

def find_split(names):
    for n in names:
        hits = [p for p in root.rglob(n) if p.is_dir() and (p / "images").exists()]
        hits += [p.parent for p in root.rglob("images") if p.is_dir() and p.parent.name.lower() == n]
        hits += [p for p in root.rglob(n) if p.is_dir() and p.parent.name == "images"]
        if hits:
            return hits[0]
    return None

train = find_split(["train"]); val = find_split(["valid", "val"]); test = find_split(["test"])
def img_dir(p):
    if p is None: return None
    return str(p / "images") if (p / "images").exists() else str(p)
print("train:", img_dir(train)); print("val:", img_dir(val)); print("test:", img_dir(test))

names = None
if yamls:
    y = yaml.safe_load(open(yamls[0]))
    names = y["names"]
    if isinstance(names, dict): names = [names[k] for k in sorted(names)]
if not names:
    ids = set()
    for f in root.rglob("*.txt"):
        for line in open(f):
            if line.strip(): ids.add(int(line.split()[0]))
    print("class ids in labels:", sorted(ids))
    names = ["Oil", "Hole", "Cutting", "Crack"][:max(ids) + 1]   # check this order against the dataset readme
print("classes:", names)

data = {"train": img_dir(train), "val": img_dir(val) or img_dir(train), "names": {i: n for i, n in enumerate(names)}}
if test: data["test"] = img_dir(test)
DATA_YAML = f"{WORK}/fabric_data.yaml"
yaml.safe_dump(data, open(DATA_YAML, "w"), sort_keys=False)
print(open(DATA_YAML).read())

## 3. Look at a few labelled samples

In [ ]:
import cv2, random, matplotlib.pyplot as plt
imgs = glob.glob(data["train"] + "/*.*")[:500]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, p in zip(axes.flat, random.sample(imgs, 8)):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    lab = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    if os.path.exists(lab):
        for line in open(lab):
            c, x, y, bw, bh = map(float, line.split()[:5])
            x1, y1 = int((x - bw / 2) * w), int((y - bh / 2) * h); x2, y2 = int((x + bw / 2) * w), int((y + bh / 2) * h)
            cv2.rectangle(im, (x1, y1), (x2, y2), (255, 0, 0), 3)
            cv2.putText(im, names[int(c)], (x1, max(20, y1 - 5)), 0, 1, (255, 0, 0), 2)
    ax.imshow(im); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Train
`yolov8n` trains in about 15 to 25 minutes on a T4. Use `yolov8s.pt` for better accuracy if you have time.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
results = model.train(data=DATA_YAML, epochs=60, imgsz=640, batch=16, patience=15,
                      project=f"{WORK}/runs", name="fabric", exist_ok=True,
                      fliplr=0.5, flipud=0.5, degrees=10, hsv_h=0.02, hsv_s=0.5, hsv_v=0.4)

## 5. Evaluate
Write the mAP50 and per class numbers on a slide. Interviewers like real metrics.

In [ ]:
best = f"{WORK}/runs/fabric/weights/best.pt"
m = YOLO(best)
metrics = m.val(data=DATA_YAML, split="test" if "test" in data else "val")
print(f"mAP50: {metrics.box.map50:.3f}   mAP50-95: {metrics.box.map:.3f}")
for i, n in enumerate(names):
    print(f"{n:10s} AP50 = {metrics.box.ap50[i]:.3f}")

In [ ]:
from IPython.display import Image, display
for f in ["confusion_matrix_normalized.png", "results.png", "PR_curve.png"]:
    p = f"{WORK}/runs/fabric/{f}"
    if os.path.exists(p): display(Image(p, width=700))

## 6. Try predictions on unseen images

In [ ]:
test_imgs = glob.glob((data.get("test") or data["val"]) + "/*.*")[:8]
preds = m.predict(test_imgs, conf=0.3, verbose=False)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, r in zip(axes.flat, preds):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Save the model for the app
Download `best.pt` from the Output panel (Kaggle) or the Files panel (Colab), then put it in the app folder as `models/best.pt`. Also save a few test images to use in the live demo.

In [ ]:
import shutil
shutil.copy(best, f"{WORK}/best.pt")
os.makedirs(f"{WORK}/demo_images", exist_ok=True)
for p in test_imgs: shutil.copy(p, f"{WORK}/demo_images/")
shutil.make_archive(f"{WORK}/demo_images", "zip", f"{WORK}/demo_images")
print("Download:", f"{WORK}/best.pt", "and", f"{WORK}/demo_images.zip")
# Colab: from google.colab import files; files.download(f"{WORK}/best.pt")